# Fluenta - Thu nghiem F5-TTS tieng Viet

**Truoc khi chay, lam theo dung thu tu sau (chi lam 1 lan):**

1. Vao **Settings** (thanh ben phai) -> **Accelerator** -> chon **GPU T4 x2**
2. Vao **Settings** -> **Internet** -> bat **On** (bat buoc de tai model tu HuggingFace)
3. Bam **+ Add Input** (goc tren ben phai) -> tim va them dataset **"vivos-vietnamese-speech-corpus-for-asr"** (dataset cong khai co san tren Kaggle, khong can tu upload gi)
4. Chay lan luot tung o (cell) tu tren xuong duoi (Shift+Enter)

Neu buoc 3 khong tim thay dataset do, ban co the tu upload 1 file .wav bat ky (giong tieng Viet ro rang, 3-8 giay) qua **+ Add Input -> Upload -> New Dataset**, roi sua duong dan `REF_AUDIO` va `REF_TEXT` o Cell 3 cho khop.

In [ ]:
# Cell 1 - Cai dat thu vien
!pip install -q f5-tts
print("Da cai xong f5-tts")

In [ ]:
# Cell 2 - Tai checkpoint tieng Viet tu HuggingFace (khoang 5.4GB, mat vai phut)
from huggingface_hub import hf_hub_download
import shutil, os

os.makedirs("/kaggle/working/f5tts_vi", exist_ok=True)

ckpt_path = hf_hub_download("hynt/F5-TTS-Vietnamese-ViVoice", "model_last.pt", local_dir="/kaggle/working/f5tts_vi")
cfg_path = hf_hub_download("hynt/F5-TTS-Vietnamese-ViVoice", "config.json", local_dir="/kaggle/working/f5tts_vi")

vocab_path = "/kaggle/working/f5tts_vi/vocab.txt"
shutil.copy(cfg_path, vocab_path)  # config.json chinh la vocab.txt, chi can doi ten

print("Checkpoint:", ckpt_path)
print("Vocab:", vocab_path)

In [ ]:
# Cell 3 - Tim 1 file audio tham chieu tu dataset VIVOS da them o buoc 3
# (neu ban tu upload dataset rieng, sua REF_AUDIO/REF_TEXT thanh cua ban)
import glob

candidates = glob.glob("/kaggle/input/**/*.wav", recursive=True)
print(f"Tim thay {len(candidates)} file .wav trong /kaggle/input")
print("Vi du vai file dau tien:")
for c in candidates[:5]:
    print(" ", c)

# CHINH LAI 2 DONG DUOI DAY cho dung file + noi dung ban chon lam giong mau:
REF_AUDIO = candidates[0] if candidates else None  # <-- sua thanh duong dan file .wav ban muon dung
REF_TEXT = "noi dung chinh xac cua file audio tren"   # <-- sua cho khop dung audio da chon

In [ ]:
# Cell 4 - Chay thu tong hop 1 vai cau co loi lap de nghe kiem tra
# Co the sua GEN_TEXTS de thu cac loai loi khac nhau
import subprocess, os

GEN_TEXTS = {
    "lap_tu": "chung toi dang dang dang song o phia bac",
    "lap_am": "hien ra t-t-tan phia xa",
    "khung_ngap_ngung": "do vay co may cung khong, , , , , , , , , , , , , , thuong dung",
    "keo_dai_am": "con duoc mammm thoi gian",
    "troi_chay": "ho dang chet nhu ruoi nhang",
}

os.makedirs("/kaggle/working/output", exist_ok=True)

for name, text in GEN_TEXTS.items():
    cmd = [
        "python", "-m", "f5_tts.infer.infer_cli",
        "--model", "F5TTS_Base",
        "--ref_audio", REF_AUDIO,
        "--ref_text", REF_TEXT,
        "--gen_text", text,
        "--vocoder_name", "vocos",
        "--vocab_file", "/kaggle/working/f5tts_vi/vocab.txt",
        "--ckpt_file", "/kaggle/working/f5tts_vi/model_last.pt",
        "--output_dir", "/kaggle/working/output",
        "--output_file", f"{name}.wav",
    ]
    print(f"=== Dang sinh: {name} ===")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("LOI:", result.stderr[-1500:])
    else:
        print("Xong ->", f"/kaggle/working/output/{name}.wav")

In [ ]:
# Cell 5 - Nghe thu truc tiep trong notebook (khong can tai ve)
import IPython.display as ipd
import glob

for f in sorted(glob.glob("/kaggle/working/output/*.wav")):
    print(f)
    ipd.display(ipd.Audio(f))

## Sau khi nghe xong

- Cac file audio nam trong `/kaggle/working/output/` - co the bam nut Download o tung o Audio player phia tren, hoac vao tab **Data > Output** ben phai de tai ca thu muc
- Neu chat luong on, bao lai ket qua de tiep tuc mo rong sinh nhieu cau hon
- Neu van gap loi (vd loi thu vien, loi tai model), copy nguyen dong bao loi gui lai de debug tiep